In [1]:
!pip install numpy langchain chromadb pypdf xmltodict transformers sentence-transformers langchain-community transformers accelerate bitsandbytes pandas --quiet

In [2]:
import os
import re
import json
import torch
import chromadb
from typing import Any
from sentence_transformers import SentenceTransformer
from accelerate import init_empty_weights, dispatch_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1, 2, 3"

In [4]:
def list_available_gpus():
    num_gpus = torch.cuda.device_count()
    print(f"Anzahl der verfügbaren GPUs: {num_gpus}")
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


def print_gpu_memory_usage(quantized_model):
    num_gpus = torch.cuda.device_count()

    print("\nGPU-Speicherverbrauch:")
    for i in range(num_gpus):
        allocated = torch.cuda.memory_allocated(i) / (1024**3)  # In GB
        reserved = torch.cuda.memory_reserved(i) / (1024**3)  # In GB
        print(f"  GPU {i}: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

    total_params = sum(p.numel() for p in quantized_model.parameters())
    total_size_gb = total_params * 2 / (1024**3)  # 2 Byte pro Parameter (bei 4-bit Quantisierung)
    print(f"\nGeschätzter Modell-Speicherverbrauch: {total_size_gb:.2f} GB")

# Verbindung zu ChromaDB

In [5]:
chroma_client = chromadb.PersistentClient(path="./chromadb")

wahlprogramme_collection = chroma_client.get_or_create_collection(name="wahlprogramme")
plenarsitzungen_collection = chroma_client.get_or_create_collection(name="plenarsitzungen")

# Embedding-Modell laden

In [6]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

# LLM laden

In [7]:
model_id = "meta-llama/Llama-3.3-70B-Instruct"

# 4-Bit-Quantisierung, perfekt für A100 mit bfloat16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Modell mit `accelerate` laden
with init_empty_weights():
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16
    )

# Tokenizer laden und anpassen
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
tokenizer.padding_side = "left"

# Sicherstellen, dass PyTorch optimiert für mehrere GPUs arbeitet
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [8]:
list_available_gpus()
print_gpu_memory_usage(quantized_model)

Anzahl der verfügbaren GPUs: 4
GPU 0: NVIDIA A100-SXM4-80GB
GPU 1: NVIDIA A100-SXM4-80GB
GPU 2: NVIDIA A100-SXM4-80GB
GPU 3: NVIDIA A100-SXM4-80GB

GPU-Speicherverbrauch:
  GPU 0: 9.80 GB allocated, 9.84 GB reserved
  GPU 1: 8.23 GB allocated, 8.27 GB reserved
  GPU 2: 8.23 GB allocated, 8.27 GB reserved
  GPU 3: 12.65 GB allocated, 12.71 GB reserved

Geschätzter Modell-Speicherverbrauch: 67.67 GB


# Funktion um mit LLM eine Antwort zu generieren

In [29]:
def query_party_analysis(
    query: str,
    tokenizer: Any,
    embedding_model: Any,
    quantized_model: Any,
    wahlprogramme_collection: Any,
    plenarsitzungen_collection: Any,
    n_results: int = 5,
    max_new_tokens: int = 1000,
    do_sample: bool = False
) -> str:

    # Embedding der Frage erstellen
    query_embedding = embedding_model.encode(query, normalize_embeddings=True).tolist()

    # Quellen aus ChromaDB laden
    wahl_results = wahlprogramme_collection.query(query_embeddings=[query_embedding], n_results=n_results)
    sitzung_results = plenarsitzungen_collection.query(query_embeddings=[query_embedding], n_results=n_results)
    
    wahlprogramme_texts = "\n\n".join(wahl_results.get("documents", [[]])[0])
    plenarsitzungen_texts = "\n\n".join(sitzung_results.get("documents", [[]])[0])

    # System Prompt
    system_message = f"""
    Analysiere die Haltung der Partei zu einem bestimmten Thema basierend auf den folgenden Informationen:

    - **Wahlprogramme:** {wahlprogramme_texts}
    - **Plenarsitzungen:** {plenarsitzungen_texts}

    **Antwortformat (keine zusätzlichen Einleitungen oder Wiederholungen):**
    
    0. Geb die Partei an und beschreibe das Thema mit einem Satz

    1. **Offizielle Position der Partei laut Wahlprogramm**  
       - Beschreibe die zentrale Position der Partei zu diesem Thema basierend auf ihrem Wahlprogramm.

    2. **Zentrale Aussagen aus den Plenarsitzungen**  
       - Fasse relevante Aussagen aus den Plenarsitzungen zusammen, die sich auf das Thema beziehen.

    3. **Vergleich zwischen Wahlprogramm und Plenarsitzungen**  
       - Identifiziere Gemeinsamkeiten und Unterschiede zwischen den programmatischen Aussagen und den realen Positionierungen in den Plenarsitzungen.

    4. **Bewertung der Partei-Aussagen**  
       - Analysiere die Konsistenz und Glaubwürdigkeit der Partei anhand der vorliegenden Informationen. Gehe auf mögliche Widersprüche oder offene Fragen ein.

    **KEINE zusätzlichen Erklärungen oder Wiederholungen der Eingabe. Beginne direkt mit Punkt 0.**
    
    Benutzte deutsche natürliche Sprache ohne Formatierung.
    -----
    """

    # Tokenisierung der Eingabe
    tokenizer.pad_token_id = tokenizer.eos_token_id 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = tokenizer(system_message, return_tensors="pt").to(device)

    # Generierung der Antwort
    output = quantized_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens, 
        do_sample=do_sample,
        repetition_penalty=1.2,
        temperature=0.7 if do_sample else None,
        top_p=0.9 if do_sample else None,
        eos_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    # Entferne alles vor dem "-----", deliter wird benutzt, da alle Eingaben sonst im Output wären
    delimiter = "-----"
    answer_start = full_output.find(delimiter)

    if answer_start != -1:
        final_output = full_output[answer_start + len(delimiter):].strip()
    else:
        final_output = full_output

    return final_output

# Einzelne Test ausgabe

In [30]:
query = "Wie ist die Meinung der AfD gegenüber Kernkraftwerken?"
response = query_party_analysis(query, tokenizer, embedding_model, quantized_model, wahlprogramme_collection, plenarsitzungen_collection)
print(response)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


0. Die Partei heißt "Alternative für Deutschland" (AfD), und das Thema betrifft deren Einstellung zur Energiepolitik, insbesondere zur Kernenergie.


1. Offizielle Position der Partei laut Wahlprogramm:
   Die AfD befürwortet die Nutzung klimafreundlicher Zukunftstechnologien wie Kernfusion und sicherer Kernkraftwerke ohne Subventionen. Sie möchte das deutsche Atomrecht von ideologischem Ballast befreien und die Wiederinbetriebnahme der vorhandenen Kernkraftwerke rechtlich ermöglichen.


2. Zentrale Aussagen aus den Plenarsitzungen:
   In den Plenarsitzungen wurde deutlich gemacht, dass die AfD sich für die Beibehaltung und den weiteren Ausbau der Kernenergie ausspricht. Es gab Aussagen, die darauf hinweisen, dass die Partei den Atomausstieg ablehnt und stattdessen eine langfristige Perspektive für die Kernenergienutzung in Deutschland sieht.


3. Vergleich zwischen Wahlprogramm und Plenarsitzungen:
   Sowohl im Wahlprogramm als auch in den Plenarsitzungen zeigt sich eine starke Unters

# JSON Datei mit den Topics der Wahlkampfprogramme laden und Anfragen stellen

In [31]:
def extract_topics_from_folder(folder_path, limit=1):
    topics_dict = {}

    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            file_path = os.path.join(folder_path, filename)

            # JSON-Datei laden
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    if "topics" in data:
                        topics_dict[filename] = [
                            ", ".join(topic["words"]) for topic in data["topics"][:limit]
                        ]
                except json.JSONDecodeError:
                    print(f"Fehler beim Laden von {filename}. Überspringe Datei.")

    return topics_dict

In [32]:
folder_path = "wahlprogramm_topics"  # Ordner mit JSON-Dateien
topics_dict = extract_topics_from_folder(folder_path)    

In [33]:
results = {}

for party, topics in topics_dict.items():
    print(f"Paratei: {party}")

    file_results = {}

    for topic in topics:
        query = f"Wie ist die Meiung der {file} zu dem Thema {topic}"
        print(query)
        
        file_results[topic] = query_party_analysis(query, tokenizer, embedding_model, quantized_model, wahlprogramme_collection, plenarsitzungen_collection)

    results[file] = file_results

output_file = "Ausgaben.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nAnalyse abgeschlossen! Ergebnisse gespeichert in {output_file}")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Datei: SPD_Wahlprogramm.pdf_topics.json
Wie ist die Meiung der SPD_Wahlprogramm.pdf_topics.json zu dem Thema menschenrechte, zusammenarbeit, verteidigungspolitik, demokratischen


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Datei: LINKE_Wahlprogramm.pdf_topics.json
Wie ist die Meiung der LINKE_Wahlprogramm.pdf_topics.json zu dem Thema gewerkschaften, unterstützen, arbeitsbedingungen, fördern


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

Datei: CDU_Wahlprogramm.pdf_topics.json
Wie ist die Meiung der CDU_Wahlprogramm.pdf_topics.json zu dem Thema altersvorsorge, alterssicherung, renteneintrittsalter, freibeträge


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Datei: FDP_Wahlprogramm.pdf_topics.json
Wie ist die Meiung der FDP_Wahlprogramm.pdf_topics.json zu dem Thema verwaltungsdigitalisierung, digitalisierung, bürokratie, innovationen


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Datei: BSW_Wahlprogramm.pdf_topics.json
Wie ist die Meiung der BSW_Wahlprogramm.pdf_topics.json zu dem Thema vernunft, gerechtigkeit, bürokratieabbau, bündnis

Analyse abgeschlossen! Ergebnisse gespeichert in output.json
